Imports and Setup

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings("ignore")

plt.style.use('seaborn-v0_8-darkgrid')


Load Local Inflation Data (FRED)

In [5]:
print("Loading local FRED CPI data...")

# Note: Standard FRED CSVs use 'DATE' as the column name. 
macro_data = pd.read_csv('../data/raw/CPIAUCSL.csv', parse_dates=['observation_date'], index_col='observation_date')

# Calculate Month-over-Month (MoM) Inflation Rate
# We multiply by 100 to get a clean percentage
macro_data['Inflation_MoM'] = macro_data['CPIAUCSL'].pct_change() * 100
macro_data = macro_data.dropna()

print(macro_data.head())


Loading local FRED CPI data...
                  CPIAUCSL  Inflation_MoM
observation_date                         
1947-02-01           21.62       0.651769
1947-03-01           22.00       1.757632
1947-04-01           22.00       0.000000
1947-05-01           21.95      -0.227273
1947-06-01           22.08       0.592255


Load Local Sentiment Data

In [7]:
print("Loading local SF Fed Sentiment Index (Excel)...")

sentiment_data = pd.read_excel(
    '../data/raw/news_sentiment_data.xlsx', 
    sheet_name=1, 
    parse_dates=['date'], 
    index_col='date', 
    engine='openpyxl'
)

# The sentiment data is daily. We resample it to a monthly average ('MS' = Month Start)
# to perfectly align with the monthly FRED inflation releases.
monthly_sentiment = sentiment_data.resample('MS').mean()

monthly_sentiment.rename(columns={'Sentiment_Score': 'News Sentiment'}, inplace=True)

# Merge datasets using an inner join so dates match perfectly
df_merged = macro_data.join(monthly_sentiment, how='inner')

# Drop any rows where either inflation or sentiment might be missing after the join
df_merged = df_merged.dropna()
print("Data successfully merged. Total aligned months:", len(df_merged))

# Save the processed, aligned dataset for modeling and analysis
df_merged.to_csv('../data/processed/macro_sentiment_aligned.csv')

Loading local SF Fed Sentiment Index (Excel)...
Data successfully merged. Total aligned months: 555
